# Non-line-of-Sight Imaging via 3D Gaussian Representation
**This is for colab**

In [ ]:
## mount my gdrive
## I will save the trained volumes on this drive
import os
from google.colab import drive
drive.mount('/content/gdrive')

TRANSIENT_VISUALIZE = False
CPU_DEBUG = False

model_save_base_dir = 'gdrive/MyDrive/Colab Notebooks/3D Tasks/NLOS-Gaussian'
if CPU_DEBUG:
    model_save_base_dir += '-CPUDEBUG'
os.makedirs(model_save_base_dir, exist_ok=True)


## Load Data

In [ ]:
!pip install gdown
!gdown --id 1kGVrFcNZZbZs0ute_roEOg5UkYeh3jRl
!unzip data_public.zip
!mv data_public/data .
!rm -rf data_public


### 3D rendering libraries.
!pip install pyvista
!pip install pyvista[jupyter]
!pip install trame ipywidgets
!pip install trimesh
!pip install open3d
!pip install ninja


### Relocation Repo
# !pip install git+https://github.com/yhy258/diff-gaussian-rasterization.git

# Check the current nvcc
!nvidia-smi
!nvcc --version

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1kGVrFcNZZbZs0ute_roEOg5UkYeh3jRl
From (redirected): https://drive.google.com/uc?id=1kGVrFcNZZbZs0ute_roEOg5UkYeh3jRl&confirm=t&uuid=49a421b9-39a0-4e5a-adf9-df7b6a6e5782
To: /content/data_public.zip
100% 467M/467M [00:04<00:00, 114MB/s]
Archive:  data_public.zip
   creating: data_public/
   creating: data_public/data/
  inflating: data_public/data/fk_bike_meas_180_min_256_preprocessed.mat  
  inflating: data_public/data/fk_discoball_meas_180_min_256_preprocessed.mat  
  inflating: data_public/data/fk_dragon_meas_180_min_256_preprocessed.mat  
  inflating: data_public/data/fk_statue_meas_180_min_256_preprocessed.mat  
  inflating: data_public/data/fk_teaser_meas_180_min_256_preprocessed.mat  
   creating: da

## BUILD CUDA SYSTEMS

In [ ]:
### Install my CUDA_renderer
!git clone https://github.com/yhy258/nlos-gaussian-renderer.git
%cd nlos-gaussian-renderer/submodules/cuda_renderer
# BUILD (about 2~5 minute)
# setup.py install is for debugging.
# !python setup.py install
!pip install .

%cd /content
print("FINISH install cuda_renderer.")
try:
    from nlos_gaussian_renderer import _C
    CUDA_AVAILABLE = True
except ImportError:
    print("Warning: CUDA extension not compiled. Please run 'python setup.py install'")
    CUDA_AVAILABLE = False

### Install Simple KNN
%cd /content
!git clone https://github.com/camenduru/simple-knn.git
%cd simple-knn
!pip install .
%cd /content
try:
    from simple_knn._C import distCUDA2 ### KNN using CUDA.rer import _C
    simple_knn_available = True
except ImportError:
    print("Warning: CUDA extension not compiled. Please run 'python setup.py install'")
    simple_knn_available = False

### Rasterizer framework for MCMC-ADC -> ERROR.
!sudo apt-get install libglm-dev
!git clone -b gs-mcmc https://github.com/shakibakh/diff-gaussian-rasterization.git
%cd diff-gaussian-rasterization

!pip install .
%cd /content

Cloning into 'nlos-gaussian-renderer'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (140/140), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 197 (delta 91), reused 84 (delta 40), pack-reused 57 (from 1)
Receiving objects: 100% (197/197), 446.60 MiB | 45.68 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/nlos-gaussian-renderer/submodules/cuda_renderer
Processing /content/nlos-gaussian-renderer/submodules/cuda_renderer
  Preparing metadata (setup.py) ... done
  Created wheel for nlos_gaussian_renderer: filename=nlos_gaussian_renderer-0.0.0-cp312-cp312-linux_x86_64.whl size=2985071 sha256=664894a645ec5b09f208d5be435e1c7b015743550b698f4a40b28257721fd340
  Stored in directory: /tmp/pip-ephem-wheel-cache-u1zjwdia/wheels/1a/e0/6a/9025d54f8dc9f2074b4451e314628089b39bf65ac1f03e70aa
Successfully built nlos_gaussian_renderer
/content
FINISH install cuda_renderer.
/content
Cloning into 'simple-knn'...
remote: Enumerating objects: 61,

## Configuration

In [ ]:
import argparse

class Config:
    def __init__(self):

        self.train = True # if this is False, only conduct evaluation

        self.rng = 0
        self.datadir = './data/zaragozadataset/zaragoza256_preprocessed.mat'
        self.dataset_type = 'zaragoza256'
        self.scene = 'zaragoza_bunny'
        self.gt_times = 100
        self.save_fig = True
        self.cuda = 0
        self.occlusion = False
        self.epoches = 1000
        self.start = 100
        self.end = 300
        self.num_sampling_points = 32
        self.expname = 'zaragoza-bunny-256'
        self.basedir = './logs'


        self.config = 'config'
        self.model_save_rel_dir = 'model'
        self.save_model_interval = 5000
        self.save_hist_fig_interval = 500
        self.print_interval = 100

        #### Gaussian Instance Init
        self.sh_degree = 3
        self.init_gaussian_num = 10000
        self.init_sample_margin = 0.1
        self.space_carving_init = False
        self.carving_volume_size = 64
        self.space_carving_ratio = 0.99
        self.scaling_modifier = 1.

        self.use_cuda_renderer = True
        self.rendering_type = 'netf'

        ## evaluation
        self.eval_resolution = 256

        if CPU_DEBUG:
            self.start = 150
            self.end = 250
            self.num_sampling_points = 4
            self.carving_volume_size = 4
            self.save_model_interval = 10
            self.init_gaussian_num = 500
            self.eval_resolution = 256

    def to_namespace(self):
        return argparse.Namespace(**self.__dict__)

class OptimizationParams:
    def __init__(self):
        self.iterations = 50_000
        self.position_lr_init = 0.00016
        self.position_lr_final = 0.0000016
        self.position_lr_delay_mult = 0.01
        self.position_lr_max_steps = 50_000
        self.feature_lr = 0.0025
        self.opacity_lr = 0.025
        self.scaling_lr = 0.005
        self.rotation_lr = 0.001
        self.exposure_lr_init = 0.01
        self.exposure_lr_final = 0.001
        self.exposure_lr_delay_steps = 0
        self.exposure_lr_delay_mult = 0.0
        self.percent_dense = 0.01
        self.lambda_dssim = 0.2

        ##### Densitification params
        self.mcmc_densification_flag = True
        self.densification_interval = 100
        self.opacity_reset_interval = 3000
        self.densify_from_iter = 500
        self.densify_until_iter = 25_000
        self.densify_grad_threshold = 0.0002
        self.cap_max = 100000


        ##### Loss coef
        self.regularization = False
        self.scale_reg = 0.01
        self.opacity_reg = 0.01
        self.depth_l1_weight_init = 1.0
        self.depth_l1_weight_final = 0.01
        self.random_background = False
        self.optimizer_type = "default"
        self.warmup_iter = 500

        ##### Indexing transient images
        self.nlos_data_random_indexing = True


        if CPU_DEBUG:
            self.mcmc_densification_flag = False

            self.iterations = 3_000
            self.position_lr_max_steps = 3_000
            self.densify_until_iter = 1_500

            self.cap_max = 10_000
            self.warmup_iter = 100

## Data Loader

In [ ]:
import numpy as np
import h5py
import scipy.io as scio

# We only consider confocal setting.
def load_zaragoza256_data(basedir):
    # nlos_data = h5py.File(basedir, 'r')
    nlos_data = scio.loadmat(basedir)

    data = np.array(nlos_data['data'])
    # data = torch.from_numpy(data)

    # E = np.sum(data,axis = 0)
    # E = E.reshape(-1)
    # plt.plot(E)
    # plt.savefig('data energy')
    deltaT = np.array(nlos_data['deltaT']).item()
    camera_position = np.array(nlos_data['cameraPosition']).reshape([-1])
    camera_grid_size = np.array(nlos_data['cameraGridSize']).reshape([-1])
    camera_grid_positions = np.array(nlos_data['cameraGridPositions'])
    camera_grid_points = np.array(nlos_data['cameraGridPoints']).reshape([-1])
    volume_position = np.array(nlos_data['hiddenVolumePosition']).reshape([-1])
    volume_size = np.max(np.array(nlos_data['hiddenVolumeSize']).reshape([-1])).item()
    c = 1

    return data, camera_position, camera_grid_size, camera_grid_positions, camera_grid_points, volume_position, volume_size, deltaT, c

if __name__ == '__main__':
    basedir = 'data/zaragozadataset/zaragoza256_preprocessed.mat'
    mat_chunk = load_zaragoza256_data(basedir)
    print("Inspecting mat_chunk:")
    keys = ['data', 'cameraPosition', "cameraGridSize",
        "cameraGridPositions",
        "cameraGridPoints",
        "hiddenVolumePosition",
        "hiddenVolumeSize", "deltaT", 'c'
        ]
    for i, (key, item) in enumerate(zip(keys, mat_chunk)):
        print(f"Key: {key}:")
        print(f"  Type: {type(item)}")
        if isinstance(item, np.ndarray):
            print(f"  Shape: {item.shape}")
        print("-" * 20)

Inspecting mat_chunk:
Key: data:
  Type: <class 'numpy.ndarray'>
  Shape: (512, 256, 256)
--------------------
Key: cameraPosition:
  Type: <class 'numpy.ndarray'>
  Shape: (3,)
--------------------
Key: cameraGridSize:
  Type: <class 'numpy.ndarray'>
  Shape: (2,)
--------------------
Key: cameraGridPositions:
  Type: <class 'numpy.ndarray'>
  Shape: (3, 65536)
--------------------
Key: cameraGridPoints:
  Type: <class 'numpy.ndarray'>
  Shape: (2,)
--------------------
Key: hiddenVolumePosition:
  Type: <class 'numpy.ndarray'>
  Shape: (3,)
--------------------
Key: hiddenVolumeSize:
  Type: <class 'float'>
--------------------
Key: deltaT:
  Type: <class 'float'>
--------------------
Key: c:
  Type: <class 'int'>
--------------------


## Spherical Harmonics funcs

In [ ]:
#  Copyright 2021 The PlenOctree Authors.
#  Redistribution and use in source and binary forms, with or without
#  modification, are permitted provided that the following conditions are met:
#
#  1. Redistributions of source code must retain the above copyright notice,
#  this list of conditions and the following disclaimer.
#
#  2. Redistributions in binary form must reproduce the above copyright notice,
#  this list of conditions and the following disclaimer in the documentation
#  and/or other materials provided with the distribution.
#
#  THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
#  AND ANY EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
#  IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE
#  ARE DISCLAIMED. IN NO EVENT SHALL THE COPYRIGHT HOLDER OR CONTRIBUTORS BE
#  LIABLE FOR ANY DIRECT, INDIRECT, INCIDENTAL, SPECIAL, EXEMPLARY, OR
#  CONSEQUENTIAL DAMAGES (INCLUDING, BUT NOT LIMITED TO, PROCUREMENT OF
#  SUBSTITUTE GOODS OR SERVICES; LOSS OF USE, DATA, OR PROFITS; OR BUSINESS
#  INTERRUPTION) HOWEVER CAUSED AND ON ANY THEORY OF LIABILITY, WHETHER IN
#  CONTRACT, STRICT LIABILITY, OR TORT (INCLUDING NEGLIGENCE OR OTHERWISE)
#  ARISING IN ANY WAY OUT OF THE USE OF THIS SOFTWARE, EVEN IF ADVISED OF THE
#  POSSIBILITY OF SUCH DAMAGE.

import torch

C0 = 0.28209479177387814
C1 = 0.4886025119029199
C2 = [
    1.0925484305920792,
    -1.0925484305920792,
    0.31539156525252005,
    -1.0925484305920792,
    0.5462742152960396
]
C3 = [
    -0.5900435899266435,
    2.890611442640554,
    -0.4570457994644658,
    0.3731763325901154,
    -0.4570457994644658,
    1.445305721320277,
    -0.5900435899266435
]
C4 = [
    2.5033429417967046,
    -1.7701307697799304,
    0.9461746957575601,
    -0.6690465435572892,
    0.10578554691520431,
    -0.6690465435572892,
    0.47308734787878004,
    -1.7701307697799304,
    0.6258357354491761,
]


def eval_sh(deg, sh, dirs):
    """
    Evaluate spherical harmonics at unit directions
    using hardcoded SH polynomials.
    Works with torch/np/jnp.
    ... Can be 0 or more batch dimensions.
    Args:
        deg: int SH deg. Currently, 0-3 supported
        sh: jnp.ndarray SH coeffs [..., C, (deg + 1) ** 2]
        dirs: jnp.ndarray unit directions [..., 3]
    Returns:
        [..., C]
    """
    assert deg <= 4 and deg >= 0
    coeff = (deg + 1) ** 2
    assert sh.shape[-1] >= coeff

    result = C0 * sh[..., 0]
    if deg > 0:
        x, y, z = dirs[..., 0:1], dirs[..., 1:2], dirs[..., 2:3]
        result = (result -
                C1 * y * sh[..., 1] +
                C1 * z * sh[..., 2] -
                C1 * x * sh[..., 3])

        if deg > 1:
            xx, yy, zz = x * x, y * y, z * z
            xy, yz, xz = x * y, y * z, x * z
            result = (result +
                    C2[0] * xy * sh[..., 4] +
                    C2[1] * yz * sh[..., 5] +
                    C2[2] * (2.0 * zz - xx - yy) * sh[..., 6] +
                    C2[3] * xz * sh[..., 7] +
                    C2[4] * (xx - yy) * sh[..., 8])

            if deg > 2:
                result = (result +
                C3[0] * y * (3 * xx - yy) * sh[..., 9] +
                C3[1] * xy * z * sh[..., 10] +
                C3[2] * y * (4 * zz - xx - yy)* sh[..., 11] +
                C3[3] * z * (2 * zz - 3 * xx - 3 * yy) * sh[..., 12] +
                C3[4] * x * (4 * zz - xx - yy) * sh[..., 13] +
                C3[5] * z * (xx - yy) * sh[..., 14] +
                C3[6] * x * (xx - 3 * yy) * sh[..., 15])

                if deg > 3:
                    result = (result + C4[0] * xy * (xx - yy) * sh[..., 16] +
                            C4[1] * yz * (3 * xx - yy) * sh[..., 17] +
                            C4[2] * xy * (7 * zz - 1) * sh[..., 18] +
                            C4[3] * yz * (7 * zz - 3) * sh[..., 19] +
                            C4[4] * (zz * (35 * zz - 30) + 3) * sh[..., 20] +
                            C4[5] * xz * (7 * zz - 3) * sh[..., 21] +
                            C4[6] * (xx - yy) * (7 * zz - 1) * sh[..., 22] +
                            C4[7] * xz * (xx - 3 * yy) * sh[..., 23] +
                            C4[8] * (xx * (xx - 3 * yy) - yy * (3 * xx - yy)) * sh[..., 24])
    return result

def RHO2SH(rho):
    return (rho - 0.5) / C0

def SH2RHO(sh):
    return sh * C0 + 0.5

## Point initializer

For the given pmin, pmax parameters, we can randomly initialize the points for the specific number of points.  
In addition, since these points are the number of initial gaussians, the number of albedos should be same with that.


In [ ]:
from tqdm import tqdm
import open3d as o3d
import trimesh


def init_rand_points(args, data_kwargs, margin=0.1, rho_scale=0.1, device='cuda'):
    # if use angular info == 0, we can conduct biased sampling toward the angular range.
    """
        pmin: (3,) [spatial, angular (phi, rho)]
        pmax: (3,) [spatial, angular (phi, rho)]
    """
    # initial gaussian num
    init_gaussian_num = args.init_gaussian_num
    # sampling rho
    rho = np.random.rand(init_gaussian_num, 1) * rho_scale
    # Sampling
    pmin, pmax = data_kwargs['pmin'], data_kwargs['pmax']
    pmin_cart, pmax_cart = pmin[:3].cpu().numpy(), pmax[:3].cpu().numpy() # 3,

    # samples = torch.rand((init_gaussian_num, 3), device=device)
    samples = np.random.rand(init_gaussian_num, 3)
    # rho = torch.rand((init_gaussian_num, 1), device=device)


    ### initialization: Avoid outside points
    modified_pmin_x = pmin_cart + np.abs(pmin_cart*margin)
    modified_pmax_x = pmax_cart - np.abs(pmax_cart*margin)
    samples = samples * (modified_pmax_x[None] - modified_pmin_x[None]) + modified_pmin_x[None] # N, 3

    return samples, rho


"""
    Employ : space carving.
"""
def detect_first_bounces(transient, threshold=1e-5):
    bins, height, width = transient.shape


    first_bounces = np.zeros((height, width))
    for y in range(height):
        for x in range(width):
            if np.sum(transient[:,y,x]) != 0:
                for b in range(1, bins, 1):
                    if transient[b,y,x] - transient[b-1,y,x]  > threshold:
                        first_bounces[y,x] = b
                        break
    return first_bounces

# This code is based on https://github.com/yfujimura/nlos-neus/blob/master/space_carving.py
def space_carving(args, data_kwargs):
    """
        data_kwargs: (Except for scalar values, the whole data is torch.Tensor)
            nlos_data:
            index: the indices for the shuffled data
            camera_grid_positions: The position of the camera grid (visible wall?); (Na x 3) : torch.Tensor
            camera_grid_size: The size of the camera grid : scalar
            volume_position: The center position of the hidden volume: (3,): torch.Tensor
            volume_size: The size of the hidden volume: Scalar
            volume_box_point: The vertex of the volume cube.
            deltaT: The discrete time interval in this setting
            c: The speed of the light
            pmin and pmax: the range of the volume coordinate
    """
    if args.scene == "zaragoza_bunny":
        nlos_file = "data/zaragozadataset/zaragoza256_preprocessed.mat"
        dataset_type = "zaragoza256"
        start = 0
        threshold = 1e-5 # we only consider this.

    camera_grid_positions = data_kwargs['camera_grid_positions']
    volume_position = data_kwargs['volume_position']
    volume_size = data_kwargs['volume_size']
    nlos_data = data_kwargs['nlos_data'].cpu().numpy()
    c, deltaT = data_kwargs['c'], data_kwargs['deltaT']

    ### shift the origin.
    camera_grid_positions = camera_grid_positions - volume_position[:, None]
    volume_position_np = np.zeros((1, 3)) # use numpy lib.
    vmin = volume_position - volume_size / 2
    vmax = volume_position + volume_size / 2

    radiuses = start + detect_first_bounces(nlos_data[start:], threshold=threshold)
    # L, M, N = nlos_data.shape
    radiuses = radiuses * c * deltaT
    radiuses = radiuses.reshape(-1,) # LMN

    unit_distance = volume_size / (args.carving_volume_size-1)
    xv = yv = zv = np.linspace(-volume_size / 2, volume_size / 2, args.carving_volume_size)

    coords = np.stack(np.meshgrid(xv, yv, zv, indexing='ij'),-1) # coords
    # coords = coords.transpose([1,0,2,3]) (If i use indexing='xy')
    coords = coords.reshape([-1,3]) # NT, 3
    coords = torch.from_numpy(coords.astype(np.float32)).to(camera_grid_positions.device)

    votes = torch.zeros(coords.shape[0])

    print("space carving...")

    total_votes = 0
    with torch.no_grad():
        for i in tqdm(range(0, camera_grid_positions.shape[1], 1)):
            if radiuses[i] > 0:
                total_votes += 1

                pt0 = camera_grid_positions[:,i]

                v = coords - pt0[None,:]
                diffs = torch.norm(v, dim=1)
                mask = torch.ones_like(diffs)
                mask[diffs < radiuses[i]] = 0
                votes[mask > 0] = votes[mask > 0] + 1

    threshold = torch.max(votes).item()*args.space_carving_ratio
    mask = torch.zeros_like(votes)
    mask[votes > threshold] = 1
    mask[votes <= threshold] = 0


    print("Complete voting.")
    # coords: NT, 3 -> Nt, 3
    # coords + volumepos : Nt, 3
    # print(volume_position.device, camera_grid_positions.device)

    # print(coords.device, mask.device, volume_position.device)
    coords2 = coords[torch.nonzero(mask, as_tuple=True)[0]] + volume_position[None] # volume_position: 1 x 3
    return coords2

def sample_from_feasible_space_jittering(args, data_kwargs, margin=0.1, rho_scale=0.1, device='cuda', exact_mesh_samping=False):
    # if use angular info == 0, we can conduct biased sampling toward the angular range.
    """
        pmin: (3,) [spatial, angular (phi, rho)]
        pmax: (3,) [spatial, angular (phi, rho)]
    """

    # initial gaussian num
    init_gaussian_num = args.init_gaussian_num
    # sampling rho
    rho = np.random.rand(init_gaussian_num, 1) * rho_scale

    # point sampling
    coords2 = space_carving(args, data_kwargs) # coords2: torch.Tensor. Nt, 3

    if exact_mesh_samping:
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(coords2.cpu().numpy())
        pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.05, max_nn=30))
        mesh_o3d, _ = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(pcd, depth=8)
        vertices = np.asarray(mesh_o3d.vertices)
        faces = np.asarray(mesh_o3d.triangles)
        mesh_trimesh = trimesh.Trimesh(vertices=vertices, faces=faces)
        samples, _ = trimesh.sample.sample_surface(mesh_trimesh, count=init_gaussian_num)
    else:
        pmin, pmax = data_kwargs['pmin'], data_kwargs['pmax']
        spacing = (pmax - pmin) / (args.carving_volume_size-1)
        spacing = spacing[:3]
        half_spacing = spacing / 2.0
        random_indices = torch.randint(0, coords2.shape[0], (init_gaussian_num,), device=coords2.device)
        base_points = coords2[random_indices] # Ng x 3

        random_offsets = (torch.rand_like(base_points) - 0.5) * 2 * half_spacing[None]
        samples = base_points + random_offsets

    return samples, rho



## Gaussian Model

In [ ]:

## misc funcs for Gaussian Model

def inverse_sigmoid(x):
    return torch.log(x/(1-x))


def strip_lowerdiag(L):
    uncertainty = torch.zeros((L.shape[0], 6), dtype=torch.float, device="cuda")

    uncertainty[:, 0] = L[:, 0, 0]
    uncertainty[:, 1] = L[:, 0, 1]
    uncertainty[:, 2] = L[:, 0, 2]
    uncertainty[:, 3] = L[:, 1, 1]
    uncertainty[:, 4] = L[:, 1, 2]
    uncertainty[:, 5] = L[:, 2, 2]
    return uncertainty

def strip_symmetric(sym):
    return strip_lowerdiag(sym)

def build_rotation(r):
    norm = torch.sqrt(r[:,0]*r[:,0] + r[:,1]*r[:,1] + r[:,2]*r[:,2] + r[:,3]*r[:,3])

    q = r / norm[:, None]

    R = torch.zeros((q.size(0), 3, 3), device=r.device)

    r = q[:, 0]
    x = q[:, 1]
    y = q[:, 2]
    z = q[:, 3]

    R[:, 0, 0] = 1 - 2 * (y*y + z*z)
    R[:, 0, 1] = 2 * (x*y - r*z)
    R[:, 0, 2] = 2 * (x*z + r*y)
    R[:, 1, 0] = 2 * (x*y + r*z)
    R[:, 1, 1] = 1 - 2 * (x*x + z*z)
    R[:, 1, 2] = 2 * (y*z - r*x)
    R[:, 2, 0] = 2 * (x*z - r*y)
    R[:, 2, 1] = 2 * (y*z + r*x)
    R[:, 2, 2] = 1 - 2 * (x*x + y*y)
    return R

def build_scaling_rotation(s, r):
    L = torch.zeros((s.shape[0], 3, 3), dtype=torch.float, device="cuda")
    R = build_rotation(r)

    L[:,0,0] = s[:,0]
    L[:,1,1] = s[:,1]
    L[:,2,2] = s[:,2]

    L = R @ L
    return L

def get_expon_lr_func(
    lr_init, lr_final, lr_delay_steps=0, lr_delay_mult=1.0, max_steps=1000000
):
    """
    Copied from Plenoxels

    Continuous learning rate decay function. Adapted from JaxNeRF
    The returned rate is lr_init when step=0 and lr_final when step=max_steps, and
    is log-linearly interpolated elsewhere (equivalent to exponential decay).
    If lr_delay_steps>0 then the learning rate will be scaled by some smooth
    function of lr_delay_mult, such that the initial learning rate is
    lr_init*lr_delay_mult at the beginning of optimization but will be eased back
    to the normal learning rate when steps>lr_delay_steps.
    :param conf: config subtree 'lr' or similar
    :param max_steps: int, the number of steps during optimization.
    :return HoF which takes step as input
    """

    def helper(step):
        if step < 0 or (lr_init == 0.0 and lr_final == 0.0):
            # Disable this parameter
            return 0.0
        if lr_delay_steps > 0:
            # A kind of reverse cosine decay.
            delay_rate = lr_delay_mult + (1 - lr_delay_mult) * np.sin(
                0.5 * np.pi * np.clip(step / lr_delay_steps, 0, 1)
            )
        else:
            delay_rate = 1.0
        t = np.clip(step / max_steps, 0, 1)
        log_lerp = np.exp(np.log(lr_init) * (1 - t) + np.log(lr_final) * t)
        return delay_rate * log_lerp

    return helper

In [ ]:
### Relocation
import torch
import math
CUDA_FLAG = torch.cuda.is_available() or CPU_DEBUG == False
print("Enable - relocation(gs-mcmc)")
if CUDA_FLAG:
    try:
        from diff_gaussian_rasterization import compute_relocation
    except:
        compute_relocation = None
        raise Exception("There is no [diff_gaussian_rasterization: relocation func]")

    N_max = 51
    binoms = torch.zeros((N_max, N_max)).float().cuda()
    for n in range(N_max):
        for k in range(n+1):
            binoms[n, k] = math.comb(n, k)

    def compute_relocation_cuda(opacity_old, scale_old, N):
        N.clamp_(min=1, max=N_max-1)
        return compute_relocation(opacity_old, scale_old, N, binoms, N_max)

Enable - relocation(gs-mcmc)


In [ ]:
import numpy as np
import torch
import torch.nn as nn
try:
    from simple_knn._C import distCUDA2 ### KNN using CUDA.
    KNN_FLAG = True
except:
    KNN_FLAG = False

class GaussianModel:
    def setup_functions(self):
        def build_covariance_from_scaling_rotation(scaling, scaling_modifier, rotation):
            L = build_scaling_rotation(scaling_modifier * scaling, rotation)
            actual_covariance = L @ L.transpose(1, 2)
            symm = strip_symmetric(actual_covariance)
            return symm

        self.scaling_activation = torch.exp
        self.scaling_inverse_activation = torch.log

        self.covariance_activation = build_covariance_from_scaling_rotation

        self.opacity_activation = torch.sigmoid
        self.inverse_opacity_activation = inverse_sigmoid

        self.rotation_activation = torch.nn.functional.normalize



    def __init__(self, args, device):
        self.args = args
        ### Albedo parameters (Spherical Harmonics or Constant)
        self.active_sh_degree = 0
        self.max_sh_degree = args.sh_degree # if we set the max_sh_degree == 0, this would be equal to the constant albedo.
        self._features_dc = torch.empty(0)
        self._features_rest = torch.empty(0)

        ### A position parameter
        self._mu = torch.empty(0)

        ### Covariance parameters
        self._scaling = torch.empty(0)
        self._rotation = torch.empty(0)

        ### Opacity of the Gaussian
        self._opacity = torch.empty(0)

        ### Non-trainable.
        self.mu_grad_accum = torch.empty(0)
        self.denom = torch.empty(0)
        self.spatial_lr_scale = 0

        ### Optimizer
        self.optimizer = None

        self.setup_functions()
        self.device = device


    def get_params(self):
        return {
            'mu': self._mu,
            'features_dc': self._features_dc,
            'features_rest': self._features_rest,
            'opacity': self._opacity,
            'scaling': self._scaling,
            'rotation': self._rotation,
            'optimizer': self.optimizer,
            'max_sh_degree': self.max_sh_degree,
            'active_sh_degree': self.active_sh_degree
        }

    def restore(self, load_path, training_args):
        print(f"Load Gaussian parameters from '{load_path}' ...")
        params = torch.load(load_path, map_location=self.device, weights_only=False)

        # 1. Learnable parameters
        self._mu = nn.Parameter(params['mu'].requires_grad_(True))
        self._features_dc = nn.Parameter(params['features_dc'].requires_grad_(True))
        self._features_rest = nn.Parameter(params['features_rest'].requires_grad_(True))
        self._opacity = nn.Parameter(params['opacity'].requires_grad_(True))
        self._scaling = nn.Parameter(params['scaling'].requires_grad_(True))
        self._rotation = nn.Parameter(params['rotation'].requires_grad_(True))

        # 2. SH degrees
        self.max_sh_degree = params['max_sh_degree']
        self.active_sh_degree = params['active_sh_degree']

        # 3. Optimizer
        if self.optimizer is None:
            # training_setup 등을 통해 옵티마이저를 먼저 생성해야 함
            print("Warning: Optimizer is not initialized. Let's initialize it ")
            self.training_setup(training_args)
        else:
            if isinstance(params['optimizer'], torch.optim.Adam):
                self.optimizer.load_state_dict(params['optimizer'].state_dict())
            else:
                self.optimizer.load_state_dict(params['optimizer'])

        print("Restoration complete")



    @property
    def get_scaling(self):
        return self.scaling_activation(self._scaling)

    @property
    def get_rotation(self):
        return self.rotation_activation(self._rotation)

    @property
    def get_mu(self):
        return self._mu

    @property
    def get_features(self):
        features_dc = self._features_dc
        features_rest = self._features_rest
        return torch.cat((features_dc, features_rest), dim=1)

    @property
    def get_features_dc(self):
        return self._features_dc

    @property
    def get_features_rest(self):
        return self._features_rest

    @property
    def get_opacity(self):
        return self.opacity_activation(self._opacity)

    def get_covariance(self, scaling_modifier = 1):
        return self.covariance_activation(self.get_scaling, scaling_modifier, self._rotation)

    def get_bboxes(self, scaling_modifier=1.0, sigma_scale=3.0):
        """
        AABB

        Args:
            scaling_modifier (float):
            sigma_scale (float): sigma scale that determines BBox size (일반적으로 3.0).

        Returns:
            torch.Tensor: (N, 2, 3) size tensor.
                          [:, 0, :] is bbox_min [x, y, z]
                          [:, 1, :] is bbox_max [x, y, z]
        """
        # 1. Get the center of the Gaussians
        mu = self.get_mu  # (N, 3)

        # 2. Diagonal factors of the Covariance matrices.
        scaling = self.get_scaling
        rotation = self.get_rotation

        # L = R @ S
        L = build_scaling_rotation(scaling_modifier * scaling, rotation)

        # Cov = L @ L.T
        actual_covariance = L @ L.transpose(1, 2)  # (N, 3, 3)

        # 3. AABB's half of extents
        # AABB extent = sigma_scale * sqrt(diag(Cov))
        diag_elements = torch.diagonal(actual_covariance, dim1=-2, dim2=-1)  # (N, 3)
        extents = sigma_scale * torch.sqrt(torch.clamp_min(diag_elements, 1e-8))  # (N, 3)

        # 4. AABB's min max coords
        bbox_min = mu - extents  # (N, 3)
        bbox_max = mu + extents  # (N, 3)

        # 5. (N, 2, 3) STACK!
        bboxes = torch.stack([bbox_min, bbox_max], dim=1)

        return bboxes

    def oneupSHdegree(self):
        if self.active_sh_degree < self.max_sh_degree:
            self.active_sh_degree += 1

    def create_params(self, points, rho, pmin, pmax, spatial_lr_scale=1.0): # rho : albedo
        self.spatial_lr_scale = spatial_lr_scale

        if isinstance(points, torch.Tensor):
            points = points.cpu()
        if isinstance(rho, torch.Tensor):
            rho = rho.cpu()
        fused_point_cloud = torch.tensor(np.asarray(points), dtype=torch.float, device=self.device)
        fused_rho = RHO2SH(torch.tensor(np.asarray(rho), dtype=torch.float, device=self.device))
        #### albedo would be represented as Spherical harmonics

        features = torch.zeros((fused_rho.shape[0], 1, (self.max_sh_degree + 1) ** 2), dtype=torch.float, device=self.device)
        features[:, :1, 0 ] = fused_rho
        features[:, 1:, 1:] = 0.0

        print("Number of points at initialisation : ", fused_point_cloud.shape[0])

        # Initialize covariances using KNN (I guess near the three points?
        if KNN_FLAG:
            dist2 = torch.clamp_min(distCUDA2(torch.from_numpy(np.asarray(points)).float().to(self.device)), 0.0000001)
        else:
            pmin_x, pmax_x = pmin[0], pmax[0]
            init_gaussian_num = points.shape[0]
            dist2 = (pmax_x - pmin_x) / (init_gaussian_num + 1e-9)
            dist2 = torch.clamp_min(torch.tensor(dist2, dtype=torch.float, device=self.device), 0.0000001)
        scales = torch.log(torch.sqrt(dist2))[...,None].repeat(fused_point_cloud.shape[0], 3)
        rots = torch.zeros((fused_point_cloud.shape[0], 4), dtype=torch.float, device=self.device)
        rots[:, 0] = 1

        opacities = self.inverse_opacity_activation(0.1 * torch.ones((fused_point_cloud.shape[0], 1), dtype=torch.float, device=self.device))

        # Initialize parameters
        self._mu = nn.Parameter(fused_point_cloud.requires_grad_(True)) # N x 3
        self._features_dc = nn.Parameter(features[:,:,0:1].transpose(1, 2).contiguous().requires_grad_(True)) # N x 1 x K
        self._features_rest = nn.Parameter(features[:,:,1:].transpose(1, 2).contiguous().requires_grad_(True))
        self._scaling = nn.Parameter(scales.requires_grad_(True)) # N by 3
        self._rotation = nn.Parameter(rots.requires_grad_(True)) # N by 4
        self._opacity = nn.Parameter(opacities.requires_grad_(True)) # N by 1

    def training_setup(self, training_args):
        self.mu_grad_accum = torch.zeros((self.get_mu.shape[0], 1), dtype=torch.float, device=self.device)
        self.denom = torch.zeros((self.get_mu.shape[0], 1), dtype=torch.float, device=self.device)

        ### Define Param-Dict
        #### We should define each parameters.
        l = [
            {'params': [self._mu], 'lr': training_args.position_lr_init * self.spatial_lr_scale, "name": "mu"},
            {'params': [self._features_dc], 'lr': training_args.feature_lr, "name": "f_dc"},
            {'params': [self._features_rest], 'lr': training_args.feature_lr / 20.0, "name": "f_rest"},
            {'params': [self._opacity], 'lr': training_args.opacity_lr, "name": "opacity"},
            {'params': [self._scaling], 'lr': training_args.scaling_lr, "name": "scaling"},
            {'params': [self._rotation], 'lr': training_args.rotation_lr, "name": "rotation"}
        ]

        self.optimizer = torch.optim.Adam(l, lr=0.0, eps=1e-15)
        self.mu_scheduler_args = get_expon_lr_func(lr_init=training_args.position_lr_init*self.spatial_lr_scale,
                                                    lr_final=training_args.position_lr_final*self.spatial_lr_scale,
                                                    lr_delay_mult=training_args.position_lr_delay_mult,
                                                    max_steps=training_args.position_lr_max_steps)

    def update_learning_rate(self, iteration):
        for param_group in self.optimizer.param_groups:
            if param_group["name"] == "mu":
                lr = self.mu_scheduler_args(iteration)
                param_group['lr'] = lr
                return lr

    def reset_opacity(self):
        opacities_new = self.inverse_opacity_activation(torch.min(self.get_opacity, torch.ones_like(self.get_opacity)*0.01))
        optimizable_tensors = self.replace_tensor_to_optimizer(opacities_new, "opacity")
        self._opacity = optimizable_tensors["opacity"]

    def replace_tensor_to_optimizer(self, tensor, name):
        optimizable_tensors = {}
        for group in self.optimizer.param_groups:
            if group["name"] == name:
                stored_state = self.optimizer.state.get(group['params'][0], None)
                stored_state["exp_avg"] = torch.zeros_like(tensor)
                stored_state["exp_avg_sq"] = torch.zeros_like(tensor)

                del self.optimizer.state[group['params'][0]]
                group["params"][0] = nn.Parameter(tensor.requires_grad_(True))
                self.optimizer.state[group['params'][0]] = stored_state

                optimizable_tensors[group["name"]] = group["params"][0]
        return optimizable_tensors



    def estimate_gaussian_pdf(self, input_points_ori, scaling_modifier=1.):
        """
        Utilized params:
            input_points_ori : absolute position [x, y, z] (Na x 3)
            self._mu : the (learnable) mean params of the gaussians (Ng x 3)
            self._scaling & self._rotation : the (learnable) cov params of the gaussians

        Output:
            Estimated Gaussian PDF
            G(x; mu, cov): (Ng x Na)
        """
        # get Quaternion
        scales = self.scaling_activation(self.get_scaling*scaling_modifier)
        rotations = self.rotation_activation(self._rotation)
        # get Mean
        mu = self.get_mu

        Ng = mu.shape[0]
        Na = input_points_ori.shape[0]

        # Expand the dimension for broadcasting
        # diff's shape: (Ng, Na, 3)
        diff = input_points_ori.unsqueeze(0) - mu.unsqueeze(1)

        # 2. Mahalanobis distance
        # Inverse Cov = RS^{-2}R where S is the diagonal scaling matrix.
        # print(f"Rotation shape: {rotations.shape} and Diff shape: {diff.shape}")
        # Rotation shape: torch.Size([100, 4]) and Diff shape: torch.Size([100, 51200, 3])
        rots = build_rotation(rotations)
        T = torch.matmul(rots.unsqueeze(1), diff.unsqueeze(-1)).squeeze(-1)

        mahalanobis_sq = torch.sum((T / scales.unsqueeze(1))**2, dim=-1) # shape: (Ng, Na)
        exponent = -0.5 * mahalanobis_sq

        # # 3. Determinant
        # # |RSS^TR^T| = |R||S||S||R|
        # cov_det = (scales[:, 0] * scales[:, 1] * scales[:, 2])**2 # Ng
        # norm_const = 1.0 / torch.sqrt((2 * torch.pi)**3 * cov_det + 1e-9) # Ng
        # pdf = norm_const.unsqueeze(1) * torch.exp(exponent) # Ng x Na
        pdf = torch.exp(exponent) # neglecting the normalization factor like 3DGS

        return pdf


    def estimate_rho_w(self, input_points_ori, current_camera_grid_positions, c, deltaT, scaling_modifier=1.0, out_separately=False):
        # input_points_ori : absolute position [x, y, z] (Na x 3) where Na = Nr * N\theta * N\phi
        # current_camera_grid_positions: (3,)
        gaussian_pdf = self.estimate_gaussian_pdf(input_points_ori, scaling_modifier) # Ng x Na
        opacity = self.get_opacity # Ng x 1

        ### Calculate rho from SHs.
        #### View-dependent albedo
        shs_view = self.get_features.transpose(1, 2).view(-1, 1, (self.max_sh_degree+1)**2)
        dir_pp = self.get_mu - current_camera_grid_positions.unsqueeze(0) # Ng, 3
        dir_pp_normalized = dir_pp / dir_pp.norm(dim=1, keepdim=True)

        sh2rho = eval_sh(self.active_sh_degree, shs_view, dir_pp_normalized)
        rho = torch.clamp_min(sh2rho + 0.5, 0.0) # Ng by 1


        density = gaussian_pdf * opacity # Ng x Na
        num_r = self.args.end - self.args.start
        density = density.view(-1, num_r, self.args.num_sampling_points ** 2) # Ng x Nr x (Ns*Ns)
        if self.args.rendering_type.lower() == 'netf':
            occlusion = torch.exp(-density * c * deltaT) # Ng x Nr x (Ns*Ns)
            # transmittance = torch.cat([torch.ones([1, occlusion.shape[2]]), occlusion + 1e-7])
            transmittance = torch.cumprod(
                torch.cat([torch.ones([occlusion.shape[0], 1, occlusion.shape[2]]), occlusion+1e-7], 1), 1
            )[:, :-1, :] # cummulative production. Ng x Nr x (Ns * Ns)
            density = density.view(-1, num_r*self.args.num_sampling_points**2) # Ng x Na
            transmittance = transmittance.view(-1, num_r*self.args.num_sampling_points**2)
            rho_density = torch.sum(density * transmittance * rho, dim=0) * c * deltaT

        elif self.args.rendering_type.lower() == 'nlos-neus':
            alpha = 1 - torch.exp(-density * c * deltaT) # Ng x Nr x (Ns*Ns)
            transmittance = torch.cumprod(
                torch.cat([torch.ones([alpha.shape[0], 1, alpha.shape[2]]), 1-alpha+1e-7], 1), 1
            )[:, :-1, :] # cummulative production. Ng x Nr x (Ns * Ns)
            alpha = alpha.view(-1, num_r*self.args.num_sampling_points**2)
            transmittance = transmittance.view(-1, num_r*self.args.num_sampling_points**2)

            alpha = 1 - torch.exp(-density * c * deltaT)
            # occlusion = torch.exp(-density * c * deltaT)
            transmittance = torch.cumprod(torch.cat([torch.ones([1, alpha.shape[1]]), 1 - alpha + 1e-7], 0), 0)[:-1, :]
            alpha = alpha.view(-1) # Na
            transmittance = transmittance.view(-1) # Na
            rho_density = torch.sum(alpha * transmittance * rho, dim=0)

        if out_separately:
            return rho_density, density.view(-1), rho # Na,
        else:
            return rho_density # Na,

    def estimate_rho_w_no_occlusion(self, input_points_ori, current_camera_grid_positions, c, deltaT, scaling_modifier=1.0, out_separately=False):
        gaussian_pdf = self.estimate_gaussian_pdf(input_points_ori, scaling_modifier) # Ng x Na
        opacity = self.get_opacity

        shs_view = self.get_features.transpose(1, 2).view(-1, 1, (self.max_sh_degree+1)**2)
        dir_pp = self.get_mu - current_camera_grid_positions.unsqueeze(0) # Ng, 3
        dir_pp_normalized = dir_pp / dir_pp.norm(dim=1, keepdim=True)

        sh2rho = eval_sh(self.active_sh_degree, shs_view, dir_pp_normalized)
        rho = torch.clamp_min(sh2rho + 0.5, 0.0) # Ng by 1

        density = torch.sum(gaussian_pdf * opacity, dim=0)
        # print(rho.shape)
        rho_density = torch.sum(gaussian_pdf * opacity * rho, dim=0) # Na

        if out_separately:
            return rho_density, density.view(-1), rho # Na,
        else:
            return rho_density # Na,

    def batch_estimate_rho_w_no_occlusion(self, input_points_ori, camera_grid_positions, c, deltaT, scaling_modifier=1.0, out_separately=False):
        # camera_grid_positions : 3, Nx*Ny
        camera_grid_positions = camera_grid_positions.permute(1, 0) # Nx*Ny, 3
        gaussian_pdf = self.estimate_gaussian_pdf(input_points_ori, scaling_modifier) # Ng x Na
        opacity = self.get_opacity

        shs_view = self.get_features.transpose(1, 2).view(-1, 1, (self.max_sh_degree+1)**2)
        dir_pp = self.get_mu.unsqueeze(1) - camera_grid_positions.unsqueeze(0) # Ng, Nx*Ny, 3
        dir_pp_normalized = dir_pp / dir_pp.norm(dim=2, keepdim=True) # Ng, Nx*Ny, 3

        sh2rho = eval_sh(self.active_sh_degree, shs_view, dir_pp_normalized) # Ng, Nx*Ny, 1
        rho = torch.clamp_min(sh2rho + 0.5, 0.0) # Ng, Nx*Ny, 1

        # gaussian_pdf : Ng x Na
        # opacity: Ng x 1
        # rho: Ng x Nx*Ny x 1

        density = torch.sum(gaussian_pdf * opacity, dim=0)
        rho_density = torch.sum(gaussian_pdf.unsqueeze(1) * opacity.unsqueeze(1) * rho.unsqueeze(2), dim=0) # (Nx*Ny), Na, 1

        if out_separately:
            return rho_density, density.view(-1), rho
        else:
            return rho_density # (Nx * Ny), Na, 1

    ### Density control based on MCMC-gs
    def cat_tensors_to_optimizer(self, tensors_dict):
        optimizable_tensors = {}
        for group in self.optimizer.param_groups:
            assert len(group["params"]) == 1
            extension_tensor = tensors_dict[group["name"]]
            stored_state = self.optimizer.state.get(group['params'][0], None)
            if stored_state is not None:

                stored_state["exp_avg"] = torch.cat((stored_state["exp_avg"], torch.zeros_like(extension_tensor)), dim=0)
                stored_state["exp_avg_sq"] = torch.cat((stored_state["exp_avg_sq"], torch.zeros_like(extension_tensor)), dim=0)

                del self.optimizer.state[group['params'][0]]
                group["params"][0] = nn.Parameter(torch.cat((group["params"][0], extension_tensor), dim=0).requires_grad_(True))
                self.optimizer.state[group['params'][0]] = stored_state

                optimizable_tensors[group["name"]] = group["params"][0]
            else:
                group["params"][0] = nn.Parameter(torch.cat((group["params"][0], extension_tensor), dim=0).requires_grad_(True))
                optimizable_tensors[group["name"]] = group["params"][0]

        return optimizable_tensors


    def densification_postfix(self, new_mu, new_features_dc, new_features_rest, new_opacities, new_scaling, new_rotation, reset_params=True):
        d = {"mu": new_mu,
        "f_dc": new_features_dc,
        "f_rest": new_features_rest,
        "opacity": new_opacities,
        "scaling" : new_scaling,
        "rotation" : new_rotation}

        optimizable_tensors = self.cat_tensors_to_optimizer(d)
        self._mu = optimizable_tensors["mu"]
        self._features_dc = optimizable_tensors["f_dc"]
        self._features_rest = optimizable_tensors["f_rest"]
        self._opacity = optimizable_tensors["opacity"]
        self._scaling = optimizable_tensors["scaling"]
        self._rotation = optimizable_tensors["rotation"]


    def replace_tensors_to_optimizer(self, inds=None):
        tensors_dict = {"mu": self._mu,
            "f_dc": self._features_dc,
            "f_rest": self._features_rest,
            "opacity": self._opacity,
            "scaling" : self._scaling,
            "rotation" : self._rotation}

        optimizable_tensors = {}
        for group in self.optimizer.param_groups:
            assert len(group["params"]) == 1
            tensor = tensors_dict[group["name"]]
            stored_state = self.optimizer.state.get(group['params'][0], None)

            if inds is not None:
                #### Make original µs' momentum as 0.
                # TODO: 이 부분 에러 발생..
                stored_state["exp_avg"][inds] = 0
                stored_state["exp_avg_sq"][inds] = 0
            else:
                stored_state["exp_avg"] = torch.zeros_like(tensor)
                stored_state["exp_avg_sq"] = torch.zeros_like(tensor)

            del self.optimizer.state[group['params'][0]]
            group["params"][0] = nn.Parameter(tensor.requires_grad_(True))
            self.optimizer.state[group['params'][0]] = stored_state

            optimizable_tensors[group["name"]] = group["params"][0]

        self._mu = optimizable_tensors["mu"]
        self._features_dc = optimizable_tensors["f_dc"]
        self._features_rest = optimizable_tensors["f_rest"]
        self._opacity = optimizable_tensors["opacity"]
        self._scaling = optimizable_tensors["scaling"]
        self._rotation = optimizable_tensors["rotation"]

        torch.cuda.empty_cache()

        return optimizable_tensors

    def _update_params(self, idxs, ratio):
        new_opacity, new_scaling = compute_relocation_cuda(
            opacity_old=self.get_opacity[idxs, 0],
            scale_old=self.get_scaling[idxs],
            N=ratio[idxs, 0] + 1
        )
        new_opacity = torch.clamp(new_opacity.unsqueeze(-1), max=1.0 - torch.finfo(torch.float32).eps, min=0.005)
        new_opacity = self.inverse_opacity_activation(new_opacity)
        new_scaling = self.scaling_inverse_activation(new_scaling.reshape(-1, 3))

        return self._mu[idxs], self._features_dc[idxs], self._features_rest[idxs], new_opacity, new_scaling, self._rotation[idxs]

    def _sample_alives(self, probs, num, alive_indices=None):
        probs = probs / (probs.sum() + torch.finfo(torch.float32).eps)
        sampled_idxs = torch.multinomial(probs, num, replacement=True)
        if alive_indices is not None:
            sampled_idxs = alive_indices[sampled_idxs]
        ratio = torch.bincount(sampled_idxs).unsqueeze(-1) # 각 sample idxs의 빈도.
        return sampled_idxs, ratio


    # Relocate Dead Gaussians to some live Gaussians!
    def relocate_gs(self, dead_mask=None):
        if dead_mask.sum() == 0:
            return

        alive_mask = ~dead_mask
        dead_indices = dead_mask.nonzero(as_tuple=True)[0]
        alive_indices = alive_mask.nonzero(as_tuple=True)[0]

        if alive_indices.shape[0] <= 0:
            return

        probs = (self.get_opacity[alive_indices, 0])
        reinit_idx, ratio = self._sample_alives(alive_indices=alive_indices, probs=probs, num=dead_indices.shape[0])

        ######## Relocation
        ######## We should compensate the accumulated Gaussians' parameters.
        (
            self._mu[dead_indices],
            self._features_dc[dead_indices],
            self._features_rest[dead_indices],
            self._opacity[dead_indices],
            self._scaling[dead_indices],
            self._rotation[dead_indices]
        ) = self._update_params(reinit_idx, ratio=ratio)

        self._opacity[reinit_idx] = self._opacity[dead_indices]
        self._scaling[reinit_idx] = self._scaling[dead_indices]

        self.replace_tensors_to_optimizer(inds=reinit_idx)

    def add_new_gs(self, cap_max):
        current_num_points = self._opacity.shape[0]
        target_num = min(cap_max, int(1.05 * current_num_points))
        num_gs = max(0, target_num - current_num_points)

        if num_gs <= 0:
            return 0

        probs = self.get_opacity.squeeze(-1)
        add_idx, ratio = self._sample_alives(probs=probs, num=num_gs)

        (
            new_mu,
            new_features_dc,
            new_features_rest,
            new_opacity,
            new_scaling,
            new_rotation
        ) = self._update_params(add_idx, ratio=ratio)

        self._opacity[add_idx] = new_opacity
        self._scaling[add_idx] = new_scaling

        self.densification_postfix(new_mu, new_features_dc, new_features_rest, new_opacity, new_scaling, new_rotation, reset_params=False)
        self.replace_tensors_to_optimizer(inds=add_idx)

        return num_gs



In [ ]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
RELOCATION_TEST = False
if RELOCATION_TEST:

    CPU_DEBUG=False # Set CPU_DEBUG to True for CPU testing
    # Test code for relocate_gs and add_new_gs
    print("Testing relocate_gs and add_new_gs...")
    torch.cuda.empty_cache()
    test_optim_args = OptimizationParams()
    test_args = Config()
    # test_device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    test_device = torch.device("cpu") # Explicitly set device to CPU

    # Create dummy data_kwargs and optim_args for testing
    test_data_kwargs = {
        'c': 1.0,
        'pmin': torch.randn(5, device=test_device),
        'pmax': torch.randn(5, device=test_device)
    }

    # Create a GaussianModel instance

    test_model = GaussianModel(test_args, test_device)


    # Initialize the model parameters with some dummy data
    test_points = torch.randn(100, 3).to(test_device)
    test_rhos = torch.rand(100, 1).to(test_device)
    test_model.create_params(test_points, test_rhos, test_data_kwargs['pmin'], test_data_kwargs['pmax'])

    # Setup optimizer (required for densification methods)
    test_model.training_setup(test_optim_args)

    initial_gaussian_count = test_model._opacity.shape[0]
    print(f"Initial Gaussian count: {initial_gaussian_count}")

    # Perform a dummy backward pass and optimizer step to initialize state
    dummy_loss = test_model.get_mu.sum() + test_model.get_features.sum() + test_model.get_opacity.sum() + test_model.get_scaling.sum() + test_model.get_rotation.sum()
    dummy_loss.backward()
    test_model.optimizer.step()
    test_model.optimizer.zero_grad()


    # Test relocate_gs
    # Create a dummy dead mask
    dead_mask = torch.zeros(initial_gaussian_count, dtype=torch.bool).to(test_device)
    dead_mask[0:10] = True # Mark first 10 gaussians as dead
    with torch.no_grad():
        print(f"Number of gaussians to relocate: {dead_mask.sum().item()}")
        test_model.relocate_gs(dead_mask=dead_mask)
        print(f"Gaussian count after relocate_gs: {test_model._opacity.shape[0]}") # Should be the same

        # Test add_new_gs
        cap_max = initial_gaussian_count + 50
        print(f"Adding new gaussians up to cap_max: {cap_max}")
        added_count = test_model.add_new_gs(cap_max=cap_max)
        print(f"Number of gaussians added: {added_count}")
        print(f"Gaussian count after add_new_gs: {test_model._opacity.shape[0]}")

    print("Testing complete.")

Testing relocate_gs and add_new_gs...
Number of points at initialisation :  100


/tmp/ipython-input-2423123939.py:205: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  dist2 = torch.clamp_min(torch.tensor(dist2, dtype=torch.float, device=self.device), 0.0000001)


Initial Gaussian count: 100
Number of gaussians to relocate: 10
Gaussian count after relocate_gs: 100
Adding new gaussians up to cap_max: 150
Number of gaussians added: 5
Gaussian count after add_new_gs: 105
Testing complete.


## Transient Image Visualization

In [ ]:
import numpy as np
import scipy.io as sio
import cv2
import os


def visualize_transient_img():
    # --- 1. Define parameters ---
    basedir = 'data/zaragozadataset'
    db_name = os.path.join(basedir, 'zaragoza256_preprocessed.mat')
    output_name = 'zaragoza256_preprocessed_rep.mp4'
    output_dir = './output_videos'

    # --- 2. Visualizer ---
    # LOAD data
    print(f"Loading data from {db_name}...")
    mat_contents = sio.loadmat(db_name)
    data = mat_contents['data']
    ### data normalization
    data = (data - np.min(data)) / (np.max(data) - np.min(data)) * 127

    print("Data loaded successfully.")

    video_writer = None
    if output_name:
        os.makedirs(output_dir, exist_ok=True)
        frame_height, frame_width = data.shape[1], data.shape[2]

        fourcc = cv2.VideoWriter_fourcc(*'mp4v')

        video_writer = cv2.VideoWriter(
            os.path.join(output_dir, output_name),
            fourcc,
            15.0,
            (frame_width, frame_height),
            isColor=False
        )

    num_frames = data.shape[0]
    print(f"Processing {num_frames} frames...")


    for i in range(num_frames):
        # i-th frame data
        I = data[i, :, :]

        I_processed = np.clip(I, 0, 255).astype(np.uint8)


        # Writing
        if video_writer is not None:
            video_writer.write(I_processed)

        if (i + 1) % 50 == 0:
            print(f"  Processed {i+1}/{num_frames} frames...")

    # Release the instance
    if video_writer is not None:
        video_writer.release()
        print(f"\nVideo saved successfully to: {os.path.join(output_dir, output_name)}")

    print("Process finished.")

if TRANSIENT_VISUALIZE:
    visualize_transient_img()

In [ ]:
"""
CUDA Renderer Autograd Function
Defines forward and backward passes for CUDA-accelerated NLOS Gaussian rendering
"""

import torch
import torch.nn as nn
from typing import Tuple, Optional

try:
    from nlos_gaussian_renderer import _C
    CUDA_RENDERER_AVAILABLE = True
except ImportError:
    CUDA_RENDERER_AVAILABLE = False
    print("Warning: CUDA renderer not available")


class CUDARenderFunction(torch.autograd.Function):
    """
    Autograd function for CUDA ray-based rendering.

    Forward: Computes rendering result from Gaussians
    Backward: Computes gradients w.r.t. Gaussian parameters
    """

    @staticmethod
    def forward(
        ctx,
        ray_origins: torch.Tensor,          # [N_rays, 3]
        ray_directions: torch.Tensor,       # [N_rays, 3]
        t_samples: torch.Tensor,            # [N_samples]
        gaussian_means: torch.Tensor,       # [N_gaussians, 3]
        gaussian_scales: torch.Tensor,      # [N_gaussians, 3]
        gaussian_rotations: torch.Tensor,   # [N_gaussians, 4]
        gaussian_opacities: torch.Tensor,   # [N_gaussians, 1]
        gaussian_features: torch.Tensor,    # [N_gaussians, K]
        camera_pos: torch.Tensor,           # [3]
        active_sh_degree: int,
        c: float,
        deltaT: float,
        scaling_modifier: float,
        use_occlusion: bool
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Forward pass: Render rays through Gaussians

        Returns:
            rho_density: [N_samples, N_rays] - Main rendering output
            density: [N_samples, N_rays] - Density field
            transmittance: [N_samples, N_rays] - Transmittance values
        """
        if not CUDA_AVAILABLE:
            raise RuntimeError("CUDA renderer not available")

        # Call CUDA forward kernel (ensure contiguous for CUDA)
        rho_density, density, transmittance, gaussian_bboxes, gaussian_filter = _C.render_rays(
            ray_origins.contiguous(),
            ray_directions.contiguous(),
            t_samples.contiguous(),
            gaussian_means.contiguous(),
            gaussian_scales.contiguous(),
            gaussian_rotations.contiguous(),
            gaussian_opacities.contiguous(),
            gaussian_features.contiguous(),
            camera_pos.contiguous(),
            active_sh_degree,
            c,
            deltaT,
            scaling_modifier,
            use_occlusion
        )

        # Save ORIGINAL tensors for backward (NOT contiguous copies!)
        ctx.save_for_backward(
            ray_origins,
            ray_directions,
            t_samples,
            gaussian_filter,
            gaussian_means,
            gaussian_scales,
            gaussian_rotations,
            gaussian_opacities,
            gaussian_features,
            camera_pos,
            rho_density,
            density,
            transmittance
        )
        ctx.active_sh_degree = active_sh_degree
        ctx.c = c
        ctx.deltaT = deltaT
        ctx.scaling_modifier = scaling_modifier
        ctx.use_occlusion = use_occlusion

        return rho_density, density, transmittance

    @staticmethod
    def backward(ctx, grad_rho_density, grad_density, grad_transmittance):
        """
        Backward pass: Compute gradients w.r.t. Gaussian parameters

        Args:
            grad_rho_density: [N_samples, N_rays] - Gradient from loss
            grad_density: [N_samples, N_rays] - Usually None
            grad_transmittance: [N_samples, N_rays] - Usually None

        Returns:
            Gradients for all forward inputs (None for non-learnable params)
        """
        # Retrieve saved tensors
        (
            ray_origins,
            ray_directions,
            t_samples,
            gaussian_filter,
            gaussian_means,
            gaussian_scales,
            gaussian_rotations, # This is the tensor saved in forward
            gaussian_opacities,
            gaussian_features,
            camera_pos,
            rho_density,
            density,
            transmittance
        ) = ctx.saved_tensors

        # Initialize gradients
        grad_gaussian_means = None
        grad_gaussian_scales = None
        grad_gaussian_rotations = None
        grad_gaussian_opacities = None
        grad_gaussian_features = None

        # Ensure input gradient tensors are contiguous
        if grad_rho_density is not None:
            grad_rho_density = grad_rho_density.contiguous()
        if grad_density is not None:
            grad_density = grad_density.contiguous()
        if grad_transmittance is not None:
            grad_transmittance = grad_transmittance.contiguous()

        grad_gaussian_means_, grad_gaussian_scales_, grad_gaussian_rotations_, grad_gaussian_opacities_, grad_gaussian_features_ \
                =_C.render_rays_backward(
                    rho_density,
                    density,
                    transmittance,
                    grad_rho_density,
                    grad_density,
                    grad_transmittance,
                    ray_origins.contiguous(),
                    ray_directions.contiguous(),
                    t_samples.contiguous(),
                    gaussian_filter.contiguous(),
                    gaussian_means.contiguous(),
                    gaussian_scales.contiguous(),
                    gaussian_rotations.contiguous(), # Pass the saved tensor
                    gaussian_opacities.contiguous(),
                    gaussian_features.contiguous(),
                    camera_pos.contiguous(),
                    ctx.active_sh_degree, ctx.c, ctx.deltaT, ctx.scaling_modifier, ctx.use_occlusion
                )


        # Only compute gradients if needed
        if ctx.needs_input_grad[3]:  # gaussian_means
            grad_gaussian_means = grad_gaussian_means_
        if ctx.needs_input_grad[4]:  # gaussian_scales
            grad_gaussian_scales = grad_gaussian_scales_
        if ctx.needs_input_grad[5]:  # gaussian_rotations
             grad_gaussian_rotations = grad_gaussian_rotations_ # Assign the gradient here
        if ctx.needs_input_grad[6]:  # gaussian_opacities
            grad_gaussian_opacities = grad_gaussian_opacities_
        if ctx.needs_input_grad[7]:  # gaussian_features
            grad_gaussian_features = grad_gaussian_features_

        # Return gradients for all inputs (None for non-differentiable)
        return (
            None,  # ray_origins
            None,  # ray_directions
            None,  # t_samples
            grad_gaussian_means,
            grad_gaussian_scales,
            grad_gaussian_rotations,
            grad_gaussian_opacities,
            grad_gaussian_features,
            None,  # camera_pos
            None,  # active_sh_degree
            None,  # c
            None,  # deltaT
            None,  # scaling_modifier
            None,  # use_occlusion
        )


class CUDARenderModule(nn.Module):
    """
    PyTorch Module wrapper for CUDA rendering with automatic differentiation.

    This provides a clean interface for using the CUDA renderer in training loops
    with full gradient support.
    """

    def __init__(self, sigma_threshold: float = 3.0):
        """
        Args:
            sigma_threshold: Threshold for Gaussian AABB computation
        """
        super().__init__()
        if not CUDA_AVAILABLE:
            raise RuntimeError("CUDA renderer not available")

        self.sigma_threshold = sigma_threshold

    def forward(
        self,
        gaussian_model,
        camera_pos: torch.Tensor,
        theta_range: Tuple[float, float],
        phi_range: Tuple[float, float],
        r_range: Tuple[float, float],
        num_theta: int,
        num_phi: int,
        num_r: int,
        c: float,
        deltaT: float,
        scaling_modifier: float = 1.0,
        use_occlusion: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Forward pass through CUDA renderer

        Args:
            gaussian_model: GaussianModel with learnable parameters
            camera_pos: [3] Camera position
            theta_range: (min, max) theta angles
            phi_range: (min, max) phi angles
            r_range: (min, max) radial distances
            num_theta: Number of theta samples
            num_phi: Number of phi samples
            num_r: Number of radial samples
            c: Speed of light
            deltaT: Time interval
            scaling_modifier: Gaussian scale modifier
            use_occlusion: Whether to use transmittance

        Returns:
            result: [num_r, num_theta, num_phi] rendered volume
            pred_histogram: [num_r] integrated histogram
        """
        device = camera_pos.device

        # Generate rays
        theta = torch.linspace(theta_range[0], theta_range[1], num_theta, device=device)
        phi = torch.linspace(phi_range[0], phi_range[1], num_phi, device=device)

        theta_grid, phi_grid = torch.meshgrid(theta, phi, indexing='ij')
        theta_flat = theta_grid.reshape(-1)
        phi_flat = phi_grid.reshape(-1)

        num_rays = theta_flat.shape[0]

        # Ray directions in Cartesian coordinates
        ray_dirs = torch.stack([
            torch.sin(theta_flat) * torch.cos(phi_flat),
            torch.sin(theta_flat) * torch.sin(phi_flat),
            torch.cos(theta_flat)
        ], dim=1)  # [num_rays, 3]

        ray_origins = camera_pos.unsqueeze(0).expand(num_rays, 3)

        # Radial samples
        t_samples = torch.linspace(r_range[0], r_range[1], num_r, device=device)

        # Get Gaussian parameters (with gradients!)
        gaussian_means = gaussian_model.get_mu
        gaussian_scales = gaussian_model._scaling
        gaussian_rotations = gaussian_model._rotation
        gaussian_opacities = gaussian_model._opacity
        gaussian_features = gaussian_model.get_features_dc.squeeze(1)

        # Call custom autograd function
        rho_density, density, transmittance = CUDARenderFunction.apply(
            ray_origins,
            ray_dirs,
            t_samples,
            gaussian_means,
            gaussian_scales,
            gaussian_rotations,
            gaussian_opacities,
            gaussian_features,
            camera_pos,
            gaussian_model.active_sh_degree,
            c,
            deltaT,
            scaling_modifier,
            use_occlusion
        )

        # Reshape and apply geometric attenuation
        result = rho_density.T.reshape(num_r, num_theta, num_phi)

        # Geometric attenuation: sin(theta) / r^2
        distance = t_samples.view(-1, 1, 1)
        theta_3d = theta_grid.unsqueeze(0)

        result = result / (distance ** 2 + 1e-8) * torch.sin(theta_3d)

        # Angular integration
        dtheta = (theta_range[1] - theta_range[0]) / num_theta
        dphi = (phi_range[1] - phi_range[0]) / num_phi

        pred_histogram = torch.sum(result, dim=(1, 2)) * dtheta * dphi

        return result, pred_histogram


def create_cuda_render_module(sigma_threshold: float = 3.0) -> Optional[CUDARenderModule]:
    """
    Factory function to create a CUDA render module

    Args:
        sigma_threshold: AABB threshold

    Returns:
        CUDARenderModule if CUDA available, None otherwise
    """
    if not CUDA_AVAILABLE:
        return None
    return CUDARenderModule(sigma_threshold=sigma_threshold)



## Helpers

In [ ]:
import torch
import time
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import math
import matplotlib.pyplot as plt
import matplotlib
from mpl_toolkits.mplot3d import Axes3D
from scipy.stats import multivariate_normal
import scipy.io
from scipy import linalg
import multiprocessing
import numpy.matlib
import pyvista as pv
# pv.set_jupyter_backend('trame')
from skimage.measure import marching_cubes
import trimesh
import open3d as o3d
from scipy.interpolate import griddata
if CUDA_RENDERER_AVAILABLE:
    CUDA_RENDERER = create_cuda_render_module()
else:
    CUDA_RENDERER = None


def save_model(args, model, current_iter):
    # save model
    model_save_rel_dir = args.model_save_rel_dir
    model_dir = os.path.join(model_save_base_dir, model_save_rel_dir)
    os.makedirs(model_dir, exist_ok=True)
    model_name = f'{model_dir}/current_iter' + str(current_iter) + '.pt'
    params = model.get_params()
    torch.save(params, model_name)
    return 0

def gaussian2volume(args, model: GaussianModel, data_kwargs, camera_pos, resolution=128, mode='voxel'):
    with torch.no_grad():
        input_points, I1, I2, num_r, dtheta, dphi, theta_min, theta_max, phi_min, phi_max = spherical_sample_histogram(args, data_kwargs, camera_pos)
        input_points_ori = input_points[:, 0:3] # spatial coordinate
        result, w, density, albedo = model.estimate_rho_w(input_points_ori, camera_pos, c=data_kwargs['c'], deltaT=data_kwargs['deltaT'], scaling_modifier=args.scaling_modifier, out_separately=True)
        irregular_w = w.cpu().numpy()
        irregular_density = density.cpu().numpy()
        irregular_albedo = albedo.cpu().numpy()
    irregular_points = input_points_ori.cpu().numpy()


    if mode.lower() == 'mesh':
        threshold = np.mean(irregular_w)
        dense_points = irregular_points[irregular_w > threshold]
        pcd = o3d.geometry.PointCloud()
        pcd.points = o3d.utility.Vector3dVector(dense_points)

        # estimate orthogonal vector
        pcd.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.01, max_nn=30))

        # Poisson surface reconstruction
        mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
            pcd, depth=8)

        # 5. Remove triangles that have low density
        vertices_to_remove = densities < np.quantile(densities, 0.01)
        mesh.remove_vertices_by_mask(vertices_to_remove)

        # 6. result save.
        o3d.io.write_point_cloud("output_point_cloud.ply", pcd)
        o3d.io.write_triangle_mesh("output_poisson_mesh.ply", mesh)



def cartesian2spherical(pt): # (x, y, z) -> (r, \theta, \phi)
    # cartesian to spherical coordinates
    # input： pt N x 3 ndarray

    spherical_pt = np.zeros(pt.shape)
    spherical_pt[:,0] = np.sqrt(np.sum(pt ** 2,axis=1))
    spherical_pt[:,1] = np.arccos(pt[:,2] / spherical_pt[:,0])
    phi_yplus = (np.arctan(pt[:,1] / (pt[:,0] + 1e-8))) * (pt[:,1] >= 0)
    phi_yplus = phi_yplus + (phi_yplus < 0).astype(np.int32) * (np.pi)
    phi_yminus = (np.arctan(pt[:,1] / (pt[:,0] + 1e-8))) * (pt[:,1] < 0)
    phi_yminus = phi_yminus + (phi_yminus > 0).astype(np.int32) * (-np.pi)
    spherical_pt[:,2] = phi_yminus + phi_yplus
    return spherical_pt

def cartesian2spherical_torch(pt):  # (x, y, z) -> (r, \theta, \phi)
    # cartesian to spherical coordinates
    # input： pt N x 3 torch.Tensor
    spherical_pt = torch.zeros(pt.shape, device=pt.device)
    r = torch.linalg.norm(pt, dim=1)
    spherical_pt[:,0] = r
    spherical_pt[:,1] = torch.acos(pt[:, 2] / r)
    spherical_pt[:,2] = torch.atan2(pt[:, 1], pt[:, 0])
    return spherical_pt


def spherical2cartesian_torch(pt): # N x 3 -> N x 3. (r, \theta, \phi) -> (x, y, z)
    cartesian_pt = torch.zeros(pt.shape, device=pt.device)
    cartesian_pt[:,0] = pt[:,0] * torch.sin(pt[:,1]) * torch.cos(pt[:,2])
    cartesian_pt[:,1] = pt[:,0] * torch.sin(pt[:,1]) * torch.sin(pt[:,2])
    cartesian_pt[:,2] = pt[:,0] * torch.cos(pt[:,1])

    return cartesian_pt


def volume_box_point(volume_position, volume_size):
    """
    args
        volume_position: The center position of the hidden volume: (3,)
        volume_size: The size of the hidden volume: Scalar
    """
    xv, yv, zv = volume_position # center position
    x = np.array([xv - volume_size / 2, xv - volume_size / 2, xv - volume_size / 2, xv - volume_size / 2, xv + volume_size / 2, xv + volume_size / 2, xv + volume_size / 2, xv + volume_size / 2])
    y = np.array([yv - volume_size / 2, yv - volume_size / 2, yv + volume_size / 2, yv + volume_size / 2, yv - volume_size / 2, yv - volume_size / 2, yv + volume_size / 2, yv + volume_size / 2])
    z = np.array([zv - volume_size / 2, zv + volume_size / 2, zv - volume_size / 2, zv + volume_size / 2, zv - volume_size / 2, zv + volume_size / 2, zv - volume_size / 2, zv + volume_size / 2])
    box = np.stack((x, y, z),axis = 1)
    return box # output: 8 by 3. The vertex of the volume cube.



# def spherical_sample_histogram(camera_grid_positions, volume_position, volume_size, deltaT, c,  num_sampling_points, start, end):
# make torch functions.
def spherical_sample_histogram(args, data_kwargs, current_camera_grid_positions):
    """
    args
        data_kwargs:
            index: the indices for the shuffled data
            camera_grid_positions: The position of the camera grid (visible wall?); (Na x 3) : torch.Tensor
            camera_grid_size: The size of the camera grid : scalar
            volume_position: The center position of the hidden volume: (3,): torch.Tensor
            volume_size: The size of the hidden volume: Scalar
            volume_box_point: The vertex of the volume cube.
            deltaT: The discrete time interval in this setting
            c: The speed of the light
            pmin and pmax: the range of the volume coordinate
    """


    ### The distance unit is µm.
    ### and the data_kwargs' parameters have actual distance values except for "index" parameter
    x0, y0, z0 = current_camera_grid_positions
    assert isinstance(current_camera_grid_positions, torch.Tensor), "current camera grid position parameter should be torch.Tensor"

    box_point = data_kwargs['volume_box_point']
    device = box_point.device

    # shift the origin of the volume to the camera from the world's origin
    box_point = box_point - current_camera_grid_positions[None, :] # 8 by 3
    # cartesian 2 spherical coordinate system
    sphere_box_point = cartesian2spherical_torch(box_point) # 8, 3 (r, \theta, \phi)
    # set the angular bound of the volume
    theta_min = torch.min(sphere_box_point[:, 1]).item()
    theta_max = torch.max(sphere_box_point[:, 1]).item()
    phi_min = torch.min(sphere_box_point[:, 2]).item()
    phi_max = torch.max(sphere_box_point[:, 2]).item()

    # make angular grid
    num_sampling_points = args.num_sampling_points
    theta = torch.linspace(theta_min, theta_max, num_sampling_points, dtype=torch.float, device=device)
    phi = torch.linspace(phi_min, phi_max, num_sampling_points, dtype=torch.float, device=device)

    dtheta = (theta_max - theta_min) / num_sampling_points
    dphi = (phi_max - phi_min) / num_sampling_points

    # make radius grid (ray distance)
    c = data_kwargs['c']
    deltaT = data_kwargs['deltaT']
    r_min = args.start * c * deltaT
    r_max = args.end * c * deltaT
    num_r = args.end - args.start
    r = torch.linspace(r_min, r_max, num_r, dtype=torch.float, device=device) # Nr

    I1 = math.floor(r_min / (c * deltaT)) # start idx
    I2 = math.ceil(r_max / (c * deltaT)) # end idx
    num_r = r.shape[0] # The number of samples.

    grid = torch.stack(torch.meshgrid(r, theta, phi), axis=-1) # Nr, Ns, Ns, 3. This tensor would be already in GPU

    spherical = grid.reshape([-1,3]) # (N, 3) where N = NrNs^2
    cartesian = spherical2cartesian_torch(spherical)
    cartesian = cartesian + current_camera_grid_positions # re-shift the center of the volume from the cam space to the world space

    # (x, y, z; theta, phi) positions for the evaluation.
    # x, y, z are the absolute position
    # theta and phi are the relative parameters for the given camera position
    cartesian = torch.cat((cartesian, spherical[:,1:3]), axis = 1).float() # x, y, z, theta, phi
    return cartesian, I1, I2, num_r, dtheta, dphi, theta_min, theta_max, phi_min, phi_max

def gaussian_transient_rendering_cuda(args, model, data_kwargs, input_points, current_camera_grid_positions, I1, I2, num_r, dtheta, dphi):
    """
    CUDA-accelerated version of gaussian_transient_rendering.

    Uses ray-based rendering with Gaussian filtering for efficient computation.
    Only relevant Gaussians are processed for each ray.
    """
    # Extract angular ranges from input_points
    theta_vals = input_points[:, 3]
    phi_vals = input_points[:, 4]

    theta_min = theta_vals.min().item()
    theta_max = theta_vals.max().item()
    phi_min = phi_vals.min().item()
    phi_max = phi_vals.max().item()

    r_min = I1 * data_kwargs['c'] * data_kwargs['deltaT']
    r_max = I2 * data_kwargs['c'] * data_kwargs['deltaT']

    # Call CUDA renderer
    """
    gaussian_model,
    camera_pos: torch.Tensor,
    theta_range: Tuple[float, float],
    phi_range: Tuple[float, float],
    r_range: Tuple[float, float],
    num_theta: int,
    num_phi: int,
    num_r: int,
    c: float,
    deltaT: float,
    scaling_modifier: float = 1.0,
    use_occlusion: bool = True
    """
    result_3d, pred_histogram = CUDA_RENDERER(
        gaussian_model=model,
        camera_pos=current_camera_grid_positions,
        theta_range=(theta_min, theta_max),
        phi_range=(phi_min, phi_max),
        r_range=(r_min, r_max),
        num_theta=args.num_sampling_points,
        num_phi=args.num_sampling_points,
        num_r=num_r,
        c=data_kwargs['c'],
        deltaT=data_kwargs['deltaT'],
        scaling_modifier=args.scaling_modifier,
        use_occlusion=args.occlusion,
    )

    # Reshape to match original format [num_r, num_angular^2]
    result = result_3d.reshape(num_r, args.num_sampling_points * args.num_sampling_points)

    # Apply the mysterious scaling factor (kept for compatibility)
    result = result * (data_kwargs['volume_position'][1] ** 2)
    pred_histogram = pred_histogram * (data_kwargs['volume_position'][1] ** 2)

    return result, pred_histogram

def gaussian_transient_rendering(args, model, data_kwargs, input_points, current_camera_grid_positions, I1, I2, num_r, dtheta, dphi):
    if hasattr(args, 'use_cuda_renderer') and args.use_cuda_renderer and CUDA_RENDERER is not None:
        return gaussian_transient_rendering_cuda(
            args, model, data_kwargs, input_points,
            current_camera_grid_positions, I1, I2, num_r, dtheta, dphi
        )
    # Result: Na (Na = Nr x Ntheta x Nphi)
    input_points_ori = input_points[:, 0:3] # spatial coordinate Na by 3
    if args.occlusion == False:
        result = model.estimate_rho_w_no_occlusion(input_points_ori, current_camera_grid_positions, c=data_kwargs['c'], deltaT=data_kwargs['deltaT'], scaling_modifier=args.scaling_modifier)
    else:
        result = model.estimate_rho_w(input_points_ori, current_camera_grid_positions, c=data_kwargs['c'], deltaT=data_kwargs['deltaT'], scaling_modifier=args.scaling_modifier)
    ### Attenuation (Confocal Setting)
    # print("The shape of the result: ", result.shape)

    result = result.reshape(num_r, args.num_sampling_points ** 2)

    with torch.no_grad():
        distance = (torch.linspace(I1, I2, num_r, dtype=torch.float, device=input_points.device) * data_kwargs['deltaT'] * data_kwargs['c']) # I1, I2 are the start and end index. I1 * deltaT * c and I2 * deltaT * c would be the real start and end distance of the ray.
        # num_r is the number of samples for radius (ray).
        distance = distance.view(-1, 1)
        distance = distance.repeat(1, args.num_sampling_points ** 2)
        Theta = input_points.view(-1, args.num_sampling_points ** 2, 5)[:, :, 3] # (Nr, Na^2, 5)

    result = result / (distance ** 2) * torch.sin(Theta) # attenuation factor. sin(theta) / r^2
    result = result * (data_kwargs['volume_position'][1] ** 2) # WHAT?? WHY?

    pred_histogram = torch.sum(result, axis=1) # summation for the angular components.
    pred_histogram = pred_histogram * dtheta * dphi # infinitesimal factor product.
    # print("Predicted histogram's shape: ", pred_histogram.shape)

    return result, pred_histogram

def compute_loss(args, model: GaussianModel, data_kwargs: dict, optim_kwargs: dict, device: torch.device):
    """
        data_kwargs:
            index: the indices for the shuffled data
            camera_grid_positions: The position of the camera grid (visible wall?); (3 x Na) : torch.Tensor
            camera_grid_size: The size of the camera grid : scalar
            volume_position: The center position of the hidden volume: (3,): torch.Tensor
            volume_size: The size of the hidden volume: Scalar
            volume_box_point: The vertex of the volume cube.
            deltaT: The discrete time interval in this setting
            c: The speed of the light
            pmin and pmax: the range of the volume coordinate
        optim_kwargs:
            prev_time: The start time of the training
            M, m, N, n : indices for the current step
            where N is the number of columns, m is the current index of rows, and n is the current index of columns.
            (j: target histogram. According to the given setting, we can accumulate the loss using the indices j
            N_iters: epochs
            criterion: the main loss function of the model
            # optimizer is in the GaussianModel instance.
    """
    ## get the current virtual camera position
    m, N, n = optim_kwargs['m'], optim_kwargs['N'], optim_kwargs['n']
    v_flatten_pos = m * N + n

    camera_grid_positions = data_kwargs['camera_grid_positions']
    current_camera_grid_positions = camera_grid_positions[:, v_flatten_pos]

    ## get volume position bound
    pmin, pmax = data_kwargs['pmin'], data_kwargs['pmax']

    with torch.no_grad():
        ### only consider the confocal setting
        # input_points: (N, 3) where N = Nr*Na^2. Nr is the number of samples for radius, and Na is the number of samples for angular components (theta and phi.)
        # and Na == args.num_sampling_points
        input_points, I1, I2, num_r, dtheta, dphi, theta_min, theta_max, phi_min, phi_max = spherical_sample_histogram(args, data_kwargs, current_camera_grid_positions)

    #### We should devise the following function
    # print("The number of r's sampling points", num_r)
    result, pred_histogram  = gaussian_transient_rendering(args, model, data_kwargs, input_points, current_camera_grid_positions, I1, I2, num_r, dtheta, dphi)
    # print(f"Thue shape of the result: {result.shape} and the shape of the input points (sampling points): {input_points.shape}")
    #### Matching the time indices
    with torch.no_grad():
        nlos_histogram = data_kwargs['nlos_data'][I1:(I1 + num_r), m, n]
        nlos_histogram = nlos_histogram * args.gt_times
    loss = optim_kwargs['criterion'](pred_histogram, nlos_histogram)
    loss_coffe = torch.mean(nlos_histogram ** 2)
    equal_loss = loss / loss_coffe

    if args.save_fig:
        # i: epoch (not iteration.)
        if (optim_kwargs['current_iter'] % args.save_hist_fig_interval == 0):
            loss_show = equal_loss.cpu().detach().numpy()
            plt.plot(nlos_histogram.cpu(), alpha=0.5, label='data')
            plt.plot(pred_histogram.cpu().detach().numpy(), alpha = 0.5, label='predicted')
            # plt.plot(pred_histogram_extra.cpu().detach().numpy(), alpha = 0.5, label='predicted extra')
            plt.legend(loc='upper right')
            # plt.title('grid position:' + str(x0) + ' ' + str(z0))
            plt.title('grid position:' + str(format(current_camera_grid_positions[0].item(), '.4f')) + ' ' + str(format(current_camera_grid_positions[2].item(), '.4f')) + ' equal loss:' + str(format(loss_show, '.8f')) + ' coffe:' + str(format(loss_coffe.cpu().detach().numpy(), '.8f')))
            os.makedirs(f'./figure/', exist_ok=True)
            plt.savefig(f'./figure/' + str(optim_kwargs['current_iter']) + '_' + str(m) + '_' + str(n))
            plt.close()

    mdic = {'nlos':nlos_histogram.cpu().detach().numpy(),'pred':pred_histogram.cpu().detach().numpy()}
    scipy.io.savemat('./loss_compare.mat', mdic)

    return loss, equal_loss

def batch_compute_loss(args, model: GaussianModel, data_kwargs: dict, optim_kwargs: dict, device: torch.device):
    camera_grid_positions = data_kwargs['camera_grid_positions'] # 3 x Na
    pmin, pmax = data_kwargs['pmin'], data_kwargs['pmax']
    # with torch.no_grad():


## Main Runner

In [ ]:
import os, sys
from math import ceil
import numpy as np
import json
import random
import itertools
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm, trange
import scipy.io
import matplotlib.pyplot as plt


def random_seed(args):
    seed = args.rng
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def cycle_random_pairs(M, N):
    """
    Generator
    """
    all_pairs = list(itertools.product(range(M), range(N)))

    while True:
        random.shuffle(all_pairs)
        for m, n in all_pairs:
            yield m, n

@torch.no_grad()
def data_shuffle(nlos_data, camera_grid_positions, device):
    L, M, N = nlos_data.shape

    nlos_data = nlos_data.reshape(L, -1)
    camera_grid_positions = torch.from_numpy(camera_grid_positions).float().to(device)
    index = torch.linspace(0, M * N - 1, M * N, dtype=torch.float, device=device).reshape(1, -1)
    full_data = torch.cat((nlos_data, camera_grid_positions, index), axis = 0)
    full_data = full_data[:,torch.randperm(full_data.size(1))]
    nlos_data = full_data[0:L,:].view(L,M,N)
    camera_grid_positions = full_data[L:-1,:]
    index = full_data[-1,:]
    del full_data
    """
    Output:
        nlos_data (torch.Tensor) (L, M, N)
        camera_grid_positions (torch.Tensor) (MN, 3)
        index (torch.Tensor) (MN,)
    """
    return nlos_data, camera_grid_positions, index


def update_lr(optimizer, args):
    # clear varible
    # update learning rate
    for param_group in optimizer.param_groups:
        if param_group['lr'] > 0.0000001:
            param_group['lr'] = param_group['lr'] * args.lr_decay
            learning_rate = param_group['lr']
            print('learning rate is updated to ',learning_rate)
    return 0

def create_model(args, data_kwargs, optim_args, device, evaluation=False):
    # Create Gaussian Model
    model = GaussianModel(args, device)

    # initialize points and rhos
    if evaluation or args.space_carving_init == False:
        # when we evaluate them, we initialize the gaussians using random init, since space-carving requires computations to some degree.
        points, rhos = init_rand_points(args, data_kwargs, rho_scale=0.2 ,margin=args.init_sample_margin, device=device)
    else:
        points, rhos = sample_from_feasible_space_jittering(args, data_kwargs, margin=args.init_sample_margin, device=device)
    # Initialize parameters.
    print(rhos.device)
    model.create_params(points, rhos, data_kwargs['pmin'], data_kwargs['pmax'])

    # Training setup (optimizer, ...)
    model.training_setup(optim_args)

    return model


def make_data_kwargs(args, device):
    print('Data Device: ', device)
    #### zaragoza data load. (I only considered zaragoza nlos data)
    nlos_data, camera_position, camera_grid_size, camera_grid_positions, camera_grid_points, volume_position, volume_size, deltaT, c = load_zaragoza256_data(args.datadir)
    ### what is the pmin and pmax? - Max, Min spatial or angular values in the target volume
    pmin = volume_position - volume_size / 2
    pmax = volume_position + volume_size / 2
    pmin = np.concatenate((pmin,np.array([0, -np.pi])), axis = 0)
    pmax = np.concatenate((pmax,np.array([np.pi, 0])), axis = 0)
    ### get volume box points
    box_point = volume_box_point(volume_position, volume_size)


    # Device
    nlos_data = torch.tensor(nlos_data, dtype=torch.float, device=device)
    pmin = torch.tensor(pmin, dtype=torch.float, device=device)
    pmax = torch.tensor(pmax, dtype=torch.float, device=device)
    camera_grid_size = torch.tensor(camera_grid_size, dtype=torch.float, device=device)
    volume_position = torch.tensor(volume_position, dtype=torch.float, device=device)
    box_point = torch.tensor(box_point, dtype=torch.float, device=device)


    ## data shuffler
    nlos_data, camera_grid_positions, index = data_shuffle(nlos_data, camera_grid_positions, device)

    print('nlos_data device: ', nlos_data.device)
    print('volume_position device: ', volume_position.device)
    print('camera_grid_positions device: ', camera_grid_positions.device)

    # make kwargs
    data_kwargs = {
        'nlos_data': nlos_data, # L x M x N
        'index': index, # MN
        'camera_grid_positions': camera_grid_positions, # 3 x MN
        'camera_grid_size': camera_grid_size, # 2,
        'volume_position': volume_position, # 3,
        'volume_size': volume_size, # float
        'volume_box_point': box_point,
        'deltaT': deltaT, # float
        'c': c, # float
        'pmin': pmin,
        'pmax': pmax
    }
    return data_kwargs, nlos_data, camera_grid_positions, index

def make_optim_kwargs(args):
    criterion = torch.nn.MSELoss(reduction='mean')

    N_iters = args.epoches
    optim_kwargs = {'criterion': criterion, 'N_iters': N_iters}
    return optim_kwargs

def warmup_learn_func(args, optim_args, model, data_kwargs, optim_kwargs, device):
    M, N = optim_kwargs['M'], optim_kwargs['N']

    pair_generator = cycle_random_pairs(M, N)
    while optim_kwargs['current_iter'] <= optim_args.warmup_iter:
        m, n = next(pair_generator)
        model.update_learning_rate(optim_kwargs['current_iter'])
        model.optimizer.zero_grad()
        optim_kwargs['m'], optim_kwargs['n'] = m, n
        loss, equal_loss = compute_loss(args, model, data_kwargs, optim_kwargs, device)
        if optim_args.regularization:
            # reg1 = optim_args.opacity_reg * torch.abs(model.get_opacity).mean()
            # reg2 = optim_args.scale_reg * torch.abs(model.get_scaling).mean()
            loss = loss + optim_args.opacity_reg * torch.abs(model.get_opacity).mean()
            loss = loss + optim_args.scale_reg * torch.abs(model.get_scaling).mean()

        # print(f"Trainsient point ({m}, {n}) Loss: ", loss)
        loss.backward()
        with torch.no_grad():
            model.optimizer.step()
            model.optimizer.zero_grad(set_to_none = True)
            ######### If we want.. we can conduct Brownian motion!
            ######### -> SGLD.

        if optim_kwargs['current_iter'] % args.print_interval == 0:
            print(optim_kwargs['current_iter'], '/', optim_kwargs['total_iter'], 'iter  ', m, '/', data_kwargs['nlos_data'].shape[1],'  ', n,'/',data_kwargs['nlos_data'].shape[2], '  histgram loss: ',loss.item())

        optim_kwargs['current_iter'] += 1

    dt = time.time()-optim_kwargs['prev_time']
    # warmup completed time
    print(f"Complete Warmup Iterations")
    print(f"Time: {dt},  The Warmup Final Loss: {loss.item()}")
    optim_kwargs['prev_time'] = time.time()
    return model, optim_kwargs

def learn_func(args, optim_args, model, data_kwargs, optim_kwargs, device, gpu_verbose=False):
    """
        data_kwargs:
            index: the indices for the shuffled data
            camera_grid_positions: The position of the camera grid (visible wall?); (Na x 3) : torch.Tensor
            camera_grid_size: The size of the camera grid : scalar
            volume_position: The center position of the hidden volume: (3,): torch.Tensor
            volume_size: The size of the hidden volume: Scalar
            volume_box_point: The vertex of the volume cube.
            deltaT: The discrete time interval in this setting
            c: The speed of the light
            pmin and pmax: the range of the volume coordinate
        optim_kwargs:
            prev_time: The start time of the training
            M, m, N, n: indices for the current step
            where N is the number of columns, m is the current index of rows, and n is the current index of columns.
            (j: target histogram. According to the given setting, we can accumulate the loss using the indices j
            N_iters: epochs
            criterion: the main loss function of the model
            total_iter: The total number of iterations
            current_iter: The current iteration
    """

    def learn_one_iter(m, n):
        vram_log = ""
        model.update_learning_rate(optim_kwargs['current_iter'])
        model.optimizer.zero_grad()
        optim_kwargs['m'], optim_kwargs['n'] = m, n
        loss, equal_loss = compute_loss(args, model, data_kwargs, optim_kwargs, device)
        if optim_args.regularization:
            # reg1 = optim_args.opacity_reg * torch.abs(model.get_opacity).mean()
            # reg2 = optim_args.scale_reg * torch.abs(model.get_scaling).mean()
            loss = loss + optim_args.opacity_reg * torch.abs(model.get_opacity).mean()
            loss = loss + optim_args.scale_reg * torch.abs(model.get_scaling).mean()
        # print(f"Trainsient point ({m}, {n}) Loss: ", loss)
        loss.backward()

        with torch.no_grad():
            model.optimizer.step()
            model.optimizer.zero_grad(set_to_none = True)
            ######### (FUTURE WORK)
            ######### If we want.. we can conduct Brownian motion!
            ######### -> SGLD.

            # if (n % 16 == 0):
            if optim_kwargs['current_iter'] % args.print_interval == 0:
                dt = time.time()-optim_kwargs['prev_time']
                if gpu_verbose:
                    # --- VRAM Monitoring ---
                    allocated_gb = torch.cuda.memory_allocated(0) / (1024**3)
                    reserved_gb = torch.cuda.memory_reserved(0) / (1024**3)
                    vram_log = f"  VRAM(Alloc/Reserved): {allocated_gb:.2f}/{reserved_gb:.2f} GB"

                print(optim_kwargs['current_iter'], '/', optim_kwargs['total_iter'], 'iter  ', m, '/', data_kwargs['nlos_data'].shape[1],'  ', n,'/',data_kwargs['nlos_data'].shape[2], '  histgram loss: ',loss.item(), 'time: ', dt, vram_log)

                # print(i,'/',optim_kwargs['N_iters'],'iter  ', m,'/', data_kwargs['nlos_data'].shape[1],'  ', n,'/',data_kwargs['nlos_data'].shape[2], '  histgram loss: ',loss.item(), 'time: ', dt, vram_log)
                optim_kwargs['prev_time'] = time.time()
                if optim_kwargs['current_iter'] == 48:
                    total_time = dt * optim_kwargs['total_iter'] / 16 / 60 / 60
                    print('total time: ', total_time, ' hours')

            if optim_kwargs['current_iter'] % args.save_model_interval == 0:
                save_model(args, model, optim_kwargs['current_iter'])

            optim_kwargs['current_iter'] += 1
            if optim_kwargs['current_iter'] % 1000:
                model.oneupSHdegree()

            if optim_args.mcmc_densification_flag:
                if optim_kwargs['current_iter'] < optim_args.densify_until_iter and optim_kwargs['current_iter'] > optim_args.densify_from_iter and optim_kwargs['current_iter'] % optim_args.densification_interval == 0:
                    dead_mask = (model.get_opacity <= 0.005).squeeze(-1)
                    model.relocate_gs(dead_mask=dead_mask)
                    model.add_new_gs(cap_max=optim_args.cap_max)


        if optim_kwargs['current_iter'] > optim_kwargs['total_iter']:
            complete = True
            return complete
        else:
            return False

    M, N = optim_kwargs['M'], optim_kwargs['N']
    if optim_args.nlos_data_random_indexing:
        while True:
            pair_generator = cycle_random_pairs(M, N)
            m, n = next(pair_generator)
            complete = learn_one_iter(m, n)
            if complete:
                return model, optim_kwargs, complete
    else:
        for m in range(0, M):
            for n in range(0, N):
                complete = learn_one_iter(m, n)
                if complete:
                    return model, optim_kwargs, complete



def train(args, optim_args, device):
    args.scaling_modifier = 1.0


    # print args.
    print('----------------------------------------------------')
    print('Loaded: ' + args.datadir)
    print('dataset_type: ' + args.dataset_type)
    print('gt_times: ' + str(args.gt_times))
    print('save_fig: ' + str(args.save_fig))
    print('cuda: ' + str(args.cuda))
    print('start: ' + str(args.start))
    print('end: ' + str(args.end))
    print('num_sampling_points: ' + str(args.num_sampling_points))
    print('carving_volume_size: ' + str(args.carving_volume_size))
    print('----------------------------------------------------')


    # Create log dir and copy the config file
    basedir = args.basedir
    expname = args.expname
    os.makedirs(os.path.join(basedir, expname), exist_ok=True)
    f = os.path.join(basedir, expname, 'args.txt')
    with open(f, 'w') as file:
        for arg in sorted(vars(args)):
            attr = getattr(args, arg)
            file.write('{} = {}\n'.format(arg, attr))

    extrapath = './model/'
    if not os.path.exists(extrapath):
        os.makedirs(extrapath)
    extrapath = './figure/'
    if not os.path.exists(extrapath):
        os.makedirs(extrapath)
    extrapath = './figure/test'
    if not os.path.exists(extrapath):
        os.makedirs(extrapath)

    # set the dictionary inputs
    """
    These [..._kwargs] are the dictionaries that contain values dynamically changing or defined in the run code.
    On the other hand, [..._args] are the arguments that contain the code settings.
        data_kwargs:
            nlos_data: NLOS data.
            index: the indices for the shuffled data
            camera_grid_positions: The position of the camera grid (visible wall?); (Na x 3) : torch.Tensor
            camera_grid_size: The size of the camera grid : scalar
            volume_position: The center position of the hidden volume: (3,): torch.Tensor
            volume_size: The size of the hidden volume: Scalar
            volume_box_point: The vertex of the volume cube.
            deltaT: The discrete time interval in this setting
            c: The speed of the light
            pmin and pmax: the range of the volume coordinate
        optim_kwargs:
            prev_time: The start time of the training
            M, m, N, n: indices for the current step
            where N is the number of columns, m is the current index of rows, and n is the current index of columns.
            (j: target histogram. According to the given setting, we can accumulate the loss using the indices j
            N_iters: epochs
            criterion: the main loss function of the model
            optimizer: the optimizer of the model
            global_step: global step
        eval_kwargs:
            coords: The target volume coordinate mesh
            axes_coords (xv, yv, zv): Each axis coordinate for the mesh
            target_volume_shape (P, Q, R): The number of pixels on each axis
            test_batchsize:
            global_step: global step
    """
    # make data_kwargs: The whole data information.
    ## cf) Why should we get the nlos_data, camera_grid_positions, index separately? (even data_kwargs contains them.)
    ## We will use the parameters to rebalance them (two-stage training).
    ## The actually used data is data_kwargs['nlos_data'], and the global variable 'nlos_data' in the train function would be used to rebalance them.
    print(f'Current Device: {device}')
    data_kwargs, nlos_data, camera_grid_positions, index = make_data_kwargs(args, device)
    print('deltaT: ' + str(data_kwargs['deltaT']))
    # Create model
    model = create_model(args, data_kwargs, optim_args, device)

    L, M, N = nlos_data.shape

    # Make optim_kwargs (criterion, ...)
    optim_kwargs = make_optim_kwargs(args)
    optim_kwargs['M'] = M
    optim_kwargs['N'] = N

    optim_kwargs['total_iter'] = optim_args.iterations
    optim_kwargs['current_iter'] = 1

    # TRAIN
    time0 = time.time()
    optim_kwargs['prev_time'] = time0
    print('Start Training!')
    model, optim_kwargs = warmup_learn_func(args, optim_args, model, data_kwargs, optim_kwargs, device)
    while True:
        model, optim_kwargs, complete = learn_func(args, optim_args, model, data_kwargs, optim_kwargs, device)

        if complete:
            break


def evaluation(args, optim_args, load_path, device):
    data_kwargs, nlos_data, camera_grid_positions, index = make_data_kwargs(args, device)

    ## center cam pos
    cam_pos = data_kwargs['camera_grid_positions']
    _, Ns = cam_pos.shape
    N = int(math.sqrt(Ns))
    middle = N//2
    m_cam_pos = cam_pos.view(N, N, 3)[middle, middle]

    model = create_model(args, data_kwargs, optim_args, device, evaluation=True)
    model.restore(load_path, optim_args)
    # gaussian2volume(model, m_cam_pos, process_batch=args.eval_proc_batch, resolution=args.eval_resolution, mode='voxel')
    gaussian2volume(args, model, data_kwargs, m_cam_pos, resolution=args.eval_resolution, mode='mesh')



if __name__=='__main__':
    optim_args = OptimizationParams()
    args = Config()
    random_seed(args)
    if torch.cuda.is_available():
        torch.set_default_tensor_type('torch.cuda.FloatTensor')
    else:
        torch.set_default_tensor_type('torch.FloatTensor')
    device = torch.device(f"cuda:{args.cuda}" if torch.cuda.is_available() else "cpu")
    torch.cuda.empty_cache()

    if args.train:
        train(args, optim_args, device)

    model_save_rel_dir = args.model_save_rel_dir
    model_dir = os.path.join(model_save_base_dir, model_save_rel_dir)
    load_path = os.path.join(model_dir, 'current_iter110.pt')
    # load_path = '/content/gdrive/MyDrive/Colab Notebooks/3D Tasks/NLOS-Gaussian/model/current_iter30000.pt'
    evaluation(args, optim_args, load_path, device)